In [9]:
# Define the query
query = """
SELECT *
FROM [dbo].[tbl_errordetails_datamodel]
where file_id > ISNULL((SELECT max(FILE_ID)
        FROM [dbo].[tbl_universe_datamodel]), 0);
"""

# Execute the query
cursor.execute(query)

# Fetch all results
rows = cursor.fetchall()

# Convert result to DataFrame
columns = [column[0] for column in cursor.description]

# Convert pyodbc.Row objects to list of tuples
rows = [tuple(row) for row in rows]

# Define schema explicitly based on the columns
schema = StructType([StructField(col, StringType(), True) for col in columns])

# Convert result to Spark DataFrame
if rows:
    df = spark.createDataFrame(rows, schema)
else:
    df = spark.createDataFrame([], schema)

df.createOrReplaceTempView('vw_tbl_errordetails_datamodel')

In [14]:
def sp_universe_datamodel():
    
    tbl_universe_datamodel = spark.sql("""SELECT
        t.file_id as file_id, 
        t.rule_no as rule_no,
        t.attribute_group as attribute_group,
        t.unique_identifier as unique_identifier,
        COUNT(distinct error_id)  as total_asset
        FROM vw_tbl_errordetails_datamodel as t 
        group by file_id, unique_identifier, rule_no, attribute_group
    """)
    tbl_universe_datamodel.createOrReplaceTempView("vw_tbl_universe_datamodel")
    tbl_universe_datamodel.toPandas().to_sql('tbl_universe_datamodel', con=engine, if_exists='append', index=False)
    
    tbl_country_universe_ref = spark.sql( """SELECT DISTINCT
            file_id,
            rule_no,
            attribute_group,                       
            sum(total_asset) as total_universe
            from vw_tbl_universe_datamodel
            GROUP BY file_id, rule_no, attribute_group
    """)
    tbl_country_universe_ref.createOrReplaceTempView("tbl_country_universe_ref")
    tbl_country_universe_ref.toPandas().to_sql('tbl_country_universe_ref', con=engine, if_exists='append', index=False)

In [15]:
sp_universe_datamodel()